# ExoPipe candidate review

Browse the Stokes I and V 1-hour transient lists. Select a candidate and
press **Load** to center SkyWidget and load the dynamic spectrum at that
detection.

JupyterLab runs on **lwacalim10**. The review apps listen on **lwacalim07**
loopback only (`127.0.0.1:8766` Stokes V, `127.0.0.1:8768` Stokes I). A laptop
tunnel to Jupyter port **8895** is separate and does not make those ports
visible to this kernel.

On **lwacalim10**, forward the two review ports. Leave **8765** alone: that
port on this host is dataset-grader.

```bash
ssh -N -f \
  -L 127.0.0.1:8766:127.0.0.1:8766 \
  -L 127.0.0.1:8768:127.0.0.1:8768 \
  lwacalim07
```

Check from the same host before the catalog cell. Both commands should print
`200`:

```bash
curl -s -o /dev/null -w '%{http_code}\n' http://127.0.0.1:8768/
curl -s -o /dev/null -w '%{http_code}\n' http://127.0.0.1:8766/
```

Run the cells below in order. The catalog cell loads the index tables only
(names and the columns on the main list). The next cell places that table
beside the sky view, spanning the notebook. The dynamic spectrum is the wide
panel underneath. Each column header filters that column. Numeric columns and
the date take a minimum and a maximum; the other columns match text. Select a
row and press **Load** to fetch that one
`/source/…` page, center on the combined position, and show the overlay at
that detection's LST and subband. Positions are not prefetched.


In [ ]:
# Edit before running if your paths or cuts differ.
import os
from pathlib import Path

# Local Path, or a str for s3:// / https:// / DOI (do not wrap URLs in Path).
# SourceReview resolves via resolve_zarr_source (validates local stores only).
#ZARR_PATH = Path("/fast/claw/V-10min-Taper-Robust-0-dec24.zarr/")
# Native ingest layout (one time per chunk). Prefer the review layout below.
ZARR_PATH = Path("/fast/claw/IVsub_1hMHz_Taper_dewarped_aligned.zarr/")
# Open also looks for _overlay and _spectrum inserted before .zarr
# (store.zarr -> store_overlay.zarr and store_spectrum.zarr). A missing
# sibling uses ZARR_PATH for that role. Generate the copies with:
# pixi run ovro-ingest rechunk "$ZARR_PATH" "${ZARR_PATH%.zarr}_overlay.zarr" \
#   --chunk-time 1 --chunk-freq 1 --chunk-lm 4096
# pixi run ovro-ingest rechunk "$ZARR_PATH" "${ZARR_PATH%.zarr}_spectrum.zarr" \
#   --chunk-time 512 --chunk-freq 0 --chunk-lm 32
#ZARR_PATH = "s3://ovro/IVsub_1hMHz_Taper_dewarped_aligned.zarr"

# None: an s3:// ZARR_PATH loads key/secret/endpoint from the rclone remote below.
# Set a dict to override rclone ("key", "secret", "client_kwargs").
ZARR_STORAGE_OPTIONS = None
# Remote name in rclone.conf. None uses the only type=s3 remote.
RCLONE_REMOTE = "osn"
RCLONE_CONFIG = Path.home() / ".config" / "rclone" / "rclone.conf"


def storage_options_from_rclone(
    remote: str | None = None,
    config_path: Path | None = None,
) -> dict:
    """Map an rclone S3 remote onto fsspec storage_options.

    Reads ``access_key_id``, ``secret_access_key``, and ``endpoint`` from
    ``~/.config/rclone/rclone.conf``. Ceph, Minio, and Other providers use
    path-style addressing. Secrets are not printed.
    """
    import configparser

    path = Path(config_path or RCLONE_CONFIG)
    parser = configparser.ConfigParser()
    if not path.is_file() or not parser.read(path):
        msg = f"rclone config not found: {path}"
        raise FileNotFoundError(msg)
    s3_remotes = [name for name in parser.sections() if parser[name].get("type") == "s3"]
    chosen = remote or RCLONE_REMOTE
    if chosen is None:
        if len(s3_remotes) != 1:
            msg = (
                "Set RCLONE_REMOTE to one of the s3 remotes in "
                f"{path}: {s3_remotes or '(none)'}"
            )
            raise ValueError(msg)
        chosen = s3_remotes[0]
    if chosen not in parser:
        msg = f"rclone remote {chosen!r} is not in {path}"
        raise KeyError(msg)
    section = parser[chosen]
    if section.get("type") != "s3":
        msg = f"rclone remote {chosen!r} has type {section.get('type')!r}, expected 's3'"
        raise ValueError(msg)
    key = section.get("access_key_id", "").strip()
    secret = section.get("secret_access_key", "").strip()
    if not key or not secret:
        msg = f"rclone remote {chosen!r} is missing access_key_id or secret_access_key"
        raise ValueError(msg)
    options: dict = {"key": key, "secret": secret}
    token = section.get("session_token", "").strip()
    if token:
        options["token"] = token
    endpoint = section.get("endpoint", "").strip()
    client_kwargs: dict = {}
    if endpoint:
        if "://" not in endpoint:
            endpoint = f"https://{endpoint}"
        client_kwargs["endpoint_url"] = endpoint
        client_kwargs["region_name"] = section.get("region", "us-east-1").strip() or "us-east-1"
    if client_kwargs:
        options["client_kwargs"] = client_kwargs
    provider = section.get("provider", "")
    if provider in {"Ceph", "Minio", "Other"}:
        options["config_kwargs"] = {"s3": {"addressing_style": "path"}}
    print(f"S3 credentials from rclone remote {chosen!r} ({path.name})")
    return options


if ZARR_STORAGE_OPTIONS is None and isinstance(ZARR_PATH, str) and ZARR_PATH.startswith("s3://"):
    ZARR_STORAGE_OPTIONS = storage_options_from_rclone()

# Optional default for the Coordinate field (edit in the review UI after launch).
COORDINATE_STRING = ""

# Fall back to NED ObjectLookup when SkyCoord.from_name fails (requires network).
USE_NED_FALLBACK = True
NED_TIMEOUT_S = 10.0

# Known source names for autocomplete. Resolved relative to cwd and notebooks/.
KNOWN_SOURCES_PATH = Path("known_sources.yaml")

PATCH_SCALE = 5.0  # patch half-width = ceil(scale * max beam FWHM in pixels)
SKY_FOV_DEG = 8.0

# Default heatmap fill: tracked pixel or patch statistics (patch_max, mad, std, …)
HEATMAP_METHOD = "dynamic_spectrum"
PATCH_FIT_MAX_REDUCED_CHI_SQUARED = 10.0

# Upper bound on Dask l/m chunks. Finer on-disk chunks, including the
# review layout (128), stay as stored. Same bound as jupiter_flux_review.ipynb.
ZARR_LM_CHUNK = 512

# Set False to scan the image centre for the first finite time index (extra I/O)
SKIP_FIRST_VALID_SKY_SCAN = True

# Polarization subset at open. Use (0, 1) for I+V stores; None or (0,) for
# Stokes-I-only (e.g. mockfm_nl_cube.zarr has polarization size 1).
POLARIZATION_ISEL = None

# Optional local Dask cluster — see Notes (Parallelism / Dask).
# Enable for patch heatmaps on large incremental stores (I-deep, 100+ times).
# Leave False for QA subsets and for dynamic_spectrum-only review.
USE_DASK_CLIENT = False
DASK_WORKERS = 6
DASK_THREADS_PER_WORKER = 1  # use 1 when DASK_PROCESSES=True (CPU workers)
DASK_MEMORY_LIMIT = "16GiB"
DASK_PROCESSES = True  # True for CPU-bound patch reduce; False for I/O-debug only

# HiPS calibration sky (Aladin Lite background). Tiles are served over HTTP from
# lwacalim09:3005 (static server with document root = HIPS_ROOT). The browser must
# reach that host:port (not /lustre/...). Override with OVRO_HIPS_HTTP_BASE / host / port.
HIPS_ROOT = Path("/lustre/pipeline/calibration/hips")
HIPS_BACKGROUND = HIPS_ROOT / "healpix_78MHz_nside2048.hips"
HIPS_HTTP_HOST = os.environ.get("OVRO_HIPS_HTTP_HOST", "localhost")
HIPS_HTTP_PORT = int(os.environ.get("OVRO_HIPS_HTTP_PORT", "3005"))
HIPS_HTTP_PREFIX = os.environ.get(
    "OVRO_HIPS_HTTP_BASE",
    f"http://{HIPS_HTTP_HOST}:{HIPS_HTTP_PORT}",
)
os.environ.setdefault("OVRO_HIPS_HTTP_BASE", HIPS_HTTP_PREFIX)
os.environ.setdefault("OVRO_HIPS_HTTP_HOST", HIPS_HTTP_HOST)
os.environ.setdefault("OVRO_HIPS_ROOT", str(HIPS_ROOT))

# Surveys for the HiPS background selector: every *.hips folder under HIPS_ROOT
# that has a properties file.
from ovro_lwa_portal.viz.hips import list_hips_surveys

HIPS_BACKGROUNDS = tuple(list_hips_surveys(HIPS_ROOT))
print(f"{len(HIPS_BACKGROUNDS)} HiPS backgrounds under {HIPS_ROOT}")

# HiPS background display range (Aladin Lite setCuts). With both BACKGROUND_CUT_*
# None, each survey uses hips_pixel_cut from its properties file (percentiles
# when the key is missing). Set both to numbers to fix the range for every survey.
HIPS_BACKGROUND_PERCENTILE_LOW = 1.0
HIPS_BACKGROUND_PERCENTILE_HIGH = 99.0
BACKGROUND_CUT_MIN = None
BACKGROUND_CUT_MAX = None
BACKGROUND_OPACITY = 1.0

# Radio overlay color scale (SkyWidget WebGL layer).
OVERLAY_COLORMAP = "magma"  # inferno, viridis, plasma, magma, grayscale
OVERLAY_STRETCH = "log"  # linear, log, sqrt, asinh
OVERLAY_PERCENTILE_LOW = 1.0
OVERLAY_PERCENTILE_HIGH = 99.9
# Optional fixed overlay vmin/vmax (set both to override per-slice percentiles).
OVERLAY_VMIN = None
OVERLAY_VMAX = None
OVERLAY_OPACITY = 1.0

# ExoPipe 1-hour review pages on lwacalim07 loopback, via the SSH forward above.
EXOPIPE_QUEUES = {
    "V": "http://127.0.0.1:8766",
    "I": "http://127.0.0.1:8768",
}


In [ ]:
import urllib.error

import panel as pn

from ovro_lwa_portal.viz.exopipe_catalog import (
    fetch_text,
    load_candidate_table,
    parse_combined_position,
    source_page_url,
)

pn.extension("tabulator")

_FORWARD = """\
ssh -N -f \\
  -L 127.0.0.1:8766:127.0.0.1:8766 \\
  -L 127.0.0.1:8768:127.0.0.1:8768 \\
  lwacalim07"""

try:
    candidates = load_candidate_table(EXOPIPE_QUEUES)
except urllib.error.URLError:
    print("ExoPipe indexes are not reachable from this kernel.")
    print("On lwacalim10, start:")
    print(_FORWARD)
    raise

print(f"{len(candidates)} candidates")
print("Select a row in the table beside the sky view, then press Load.")


In [ ]:
from ovro_lwa_portal.viz.exopipe_catalog import (
    CandidateTabulator,
    candidate_tabulator_filters,
    parse_source_detections,
    peak_detection,
    subband_mhz_for_overlay,
)
from ovro_lwa_portal.viz.source_review_app import (
    SourceReview,
    SourceReviewConfig,
    configure_source_review_notebook,
)
from ovro_lwa_portal.viz.source_review_data import match_detection_slice

configure_source_review_notebook()

review = SourceReview(
    ZARR_PATH,
    coordinate_string=COORDINATE_STRING,
    known_sources_path=KNOWN_SOURCES_PATH,
    patch_scale=PATCH_SCALE,
    sky_fov_deg=SKY_FOV_DEG,
    patch_fit_max_reduced_chi_squared=PATCH_FIT_MAX_REDUCED_CHI_SQUARED,
    heatmap_method=HEATMAP_METHOD,
    storage_options=ZARR_STORAGE_OPTIONS,
    config=SourceReviewConfig(
        zarr_lm_chunk=ZARR_LM_CHUNK,
        polarization_isel=POLARIZATION_ISEL,
        skip_first_valid_sky_scan=SKIP_FIRST_VALID_SKY_SCAN,
        use_ned_fallback=USE_NED_FALLBACK,
        ned_timeout_s=NED_TIMEOUT_S,
        hips_root=HIPS_ROOT,
        hips_background=HIPS_BACKGROUND,
        hips_http_prefix=HIPS_HTTP_PREFIX,
        hips_backgrounds=HIPS_BACKGROUNDS,
        hips_background_percentile_low=HIPS_BACKGROUND_PERCENTILE_LOW,
        hips_background_percentile_high=HIPS_BACKGROUND_PERCENTILE_HIGH,
        background_cut_min=BACKGROUND_CUT_MIN,
        background_cut_max=BACKGROUND_CUT_MAX,
        background_opacity=BACKGROUND_OPACITY,
        overlay_colormap=OVERLAY_COLORMAP,
        overlay_stretch=OVERLAY_STRETCH,
        overlay_percentile_low=OVERLAY_PERCENTILE_LOW,
        overlay_percentile_high=OVERLAY_PERCENTILE_HIGH,
        overlay_vmin=OVERLAY_VMIN,
        overlay_vmax=OVERLAY_VMAX,
        overlay_opacity=OVERLAY_OPACITY,
    ),
)

_position_cache: dict[tuple[str, str], tuple[str, str]] = {}

# editors=None keeps cells read-only. The table is built here, after
# configure_source_review_notebook(), so its comm is the same document as
# the sky view. Header filters run in the browser: text columns match a
# substring, and numeric columns and the date take a min and a max.
header_filters, filter_configuration = candidate_tabulator_filters(candidates.columns)
candidates_table = CandidateTabulator(
    candidates,
    show_index=False,
    sizing_mode="stretch_width",
    height=620,
    header_filters=header_filters,
    configuration=filter_configuration,
    hidden_columns=["base_url"],
    editors={column: None for column in candidates.columns},
    pagination="local",
    page_size=50,
    selectable=1,
)

_chosen: dict[str, object] = {"row": None}


def _on_candidate_click(_event: object) -> None:
    row = candidates_table.clicked_row
    if row is None:
        review._log("WARNING: Select a visible row, then press Load.")
        return
    _chosen["row"] = row
    review._log(f"Selected {row.stokes} {row.source}. Press Load.")


def _detection_slice(html: str) -> tuple[int | None, int | None]:
    """Time and frequency indices for the brightest detection on this page."""
    detections = parse_source_detections(html)
    if (
        not detections
        or review._dataset is None
        or review._lst_hours is None
        or review._freq_mhz is None
    ):
        return None, None
    peak = peak_detection(detections)
    band_mhz, band_note = subband_mhz_for_overlay(peak, detections)
    time_idx, freq_idx, slice_note = match_detection_slice(
        review._dataset.coords["time"].values,
        review._freq_mhz,
        review._lst_hours,
        on_date=peak.on_date,
        lst_hour=peak.lst_hour,
        band_mhz=band_mhz,
    )
    review._log(f"{peak.band} SNR {peak.snr:.1f}: {band_note}. Store sample {slice_note}.")
    return time_idx, freq_idx


def _load_selected(_event: object | None = None) -> None:
    row = _chosen["row"]
    if row is None:
        review._log("WARNING: Select a visible row, then press Load.")
        return
    key = (str(row.base_url), str(row.source))
    review._log(f"Loading {row.stokes} {row.source}…")
    try:
        cached = _position_cache.get(key)
        if cached is None:
            html = fetch_text(source_page_url(key[0], key[1]))
            degrees = parse_combined_position(html)
            _position_cache[key] = (degrees, html)
        else:
            degrees, html = cached
        time_idx, freq_idx = _detection_slice(html)
        review.review_catalog_position(
            degrees,
            name=str(row.source),
            stokes=str(row.stokes),
            time_idx=time_idx,
            freq_idx=freq_idx,
        )
    except (urllib.error.URLError, ValueError) as exc:
        review._log(f"WARNING: Could not open {key[1]}: {exc}")


candidates_table.on_click(_on_candidate_click)
load_button = pn.widgets.Button(name="Load", button_type="primary", width=120)
load_button.on_click(_load_selected)

review._layout.max_width = None
controls, status, _sky, _heatmap, activity = list(review._layout.objects)
catalog = pn.Column(
    pn.pane.Markdown("**Candidates** — select one row, then **Load**."),
    load_button,
    candidates_table,
    sizing_mode="stretch_width",
)
review._layout.objects = [
    controls,
    status,
    pn.Row(catalog, review._sky_pane, sizing_mode="stretch_width"),
    review._heatmap_pane,
    activity,
]
review.panel
